# Main-figure source-data validation

This notebook loads the saved numerical sources for the main-text plots and redraws them with plain Matplotlib styling. It is intended as a visual data check, not as a reproduction of the paper layout. Non-numerical schematics and microscopy panels are omitted.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import numpy as np
import matplotlib.pyplot as plt

def find_data_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'tfln_paths.py').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook inside the code repository.')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

ROOT = find_data_root()
sys.path.insert(0, str(ROOT))
from tfln_paths import DATA_ROOT
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True, 'grid.alpha': 0.25})
print('Data root located.')

## Figure 1g — calculated cavity reflection amplitude and phase

In [ ]:
p = DATA_ROOT / 'fig1/fig1g/cavity_reflection_K1_state1.csv'
d = np.genfromtxt(p, delimiter=',', names=True)
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(d['detuning_pm'], d['reflection_amplitude'], color='black', label='Reflection amplitude')
ax2.plot(d['detuning_pm'], d['phase_rad'], color='tab:gray', label='Phase')
ax1.set(xlabel='Wavelength detuning (pm)', ylabel='Reflection amplitude', title='Fig. 1g source data')
ax2.set_ylabel('Phase (rad)')
fig.tight_layout()
print(f'{len(d):,} samples; amplitude range {d["reflection_amplitude"].min():.4f}–{d["reflection_amplitude"].max():.4f}')

## Figure 1h — ideal three-port reachable region

In [ ]:
reach = load_module('reachable_region', ROOT / 'figures/main/plot_reachable_region.py')
t = np.linspace(0, 2*np.pi, 901)
power = reach.boundary_power(t)
xy = reach.barycentric_xy(power)
triangle_power = np.array([[1.,0.,0.], [0.,1.,0.], [0.,0.,1.], [1.,0.,0.]])
triangle_xy = reach.barycentric_xy(triangle_power)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(triangle_xy[:,0], triangle_xy[:,1], color='0.35', linewidth=1.6)
ax.fill(xy[:,0], xy[:,1], alpha=0.42, color='#58A9E1')
ax.plot(xy[:,0], xy[:,1], color='#075A9C', linewidth=2.2)
ax.scatter(xy[::25,0], xy[::25,1], s=30, color='#075A9C', edgecolor='white', linewidth=0.5, zorder=3)
ax.scatter(triangle_xy[:3,0], triangle_xy[:3,1], s=70, color=['#D62728','#2CA02C','#1F77B4'], edgecolor='white', linewidth=0.8, zorder=4)
for (vx, vy), label in zip(triangle_xy[:3], ['P1 = 1', 'P2 = 1', 'P3 = 1']):
    ax.annotate(label, (vx, vy), xytext=(6, 6), textcoords='offset points', weight='bold')
ax.set(aspect='equal', title='Fig. 1h reachable-region boundary', xlabel='Barycentric x', ylabel='Barycentric y')
print(f'{len(power)} boundary samples; max conservation error = {np.max(np.abs(power.sum(axis=1)-1)):.3e}')

## Figure 3a — measured transmission and reflection spectra

In [ ]:
p = DATA_ROOT / 'fig3/fig3a/fig3a_plotted_source_data.csv'
d = np.genfromtxt(p, delimiter=',', names=True, dtype=None, encoding='utf-8')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(d['wavelength_nm'], d['transmission_plotted_au'], label='Transmission', color='black')
axes[0].plot(d['wavelength_nm'], d['reflection_plotted_au'], label='Reflection', color='tab:gray')
axes[0].set(xlabel='Wavelength (nm)', ylabel='Normalized intensity', title='Fig. 3a full plotted range')
axes[0].legend()
inset = d['shown_in_inset']
axes[1].plot(d['inset_delta_wavelength_pm'][inset], d['transmission_plotted_au'][inset], label='Transmission', color='black')
axes[1].plot(d['inset_delta_wavelength_pm'][inset], d['reflection_plotted_au'][inset], label='Reflection', color='tab:gray')
axes[1].set(xlabel='Detuning from fitted center (pm)', ylabel='Normalized intensity', title='Fig. 3a inset samples')
axes[1].legend()
fig.tight_layout()
print(f'{len(d):,} main-trace samples; {np.count_nonzero(inset):,} inset samples')

## Figure 3b — measured electro-optic tuning

In [ ]:
p = DATA_ROOT / 'fig3/fig3b/fig3b_effective_tuning_source_data.csv'
d = np.genfromtxt(p, delimiter=',', names=True)
fig, ax = plt.subplots()
ax.scatter(d['voltage_v'], d['measured_frequency_detuning_ghz'], color='black', label='Measured')
ax.plot(d['voltage_v'], d['empirical_fit_frequency_detuning_ghz'], label='Empirical fit')
ax.plot(d['voltage_v'], d['linear_fit_frequency_detuning_ghz'], '--', label='Linear fit')
ax.set(xlabel='Applied voltage (V)', ylabel='Frequency detuning (GHz)', title='Fig. 3b source data')
ax.legend()
fig.tight_layout()
print(f'{len(d)} measured voltage points')

## Figure 3c — normalized electro-optic response and third-order fit

In [ ]:
bw = load_module('bandwidth_fit', ROOT / 'figures/main/fit_bandwidth.py')
dut = DATA_ROOT / 'fig3/fig3c/a46_bandwidth_13dbm_2605111807.s2p'
through = DATA_ROOT / 'fig3/fig3c/through_meas_2605111715.s2p'
frequency, response = bw.through_corrected_response(dut, through)
fit3, prediction3 = bw.fit_fixed_order(frequency, response, 3)
response = response - fit3.offset_db
prediction3 = prediction3 - fit3.offset_db
crossing = bw.direct_crossing(frequency, response)
fig, ax = plt.subplots()
ax.plot(frequency, response, 'o-', ms=3, label='Measurement')
ax.plot(frequency, prediction3, '--', label='Third-order fit')
ax.axhline(-3, color='0.6', ls=':')
ax.set(xlabel='Frequency (GHz)', ylabel='Normalized response (dB)', title='Fig. 3c source data')
ax.legend()
fig.tight_layout()
print(f'{len(frequency)} samples; direct -3 dB crossing {crossing:.3f} GHz; fitted bandwidth {fit3.bandwidth_3db_ghz:.3f} GHz')

## Figure 4 setup

Load the experimental cube, reciprocal-model cube, and stroboscopic capture. The displayed RGB mapping is output 1 (CH2) → red, output 2 (CH3) → green, output 3 (CH4) → blue.

In [ ]:
cube_dir = DATA_ROOT / 'fig4/voltage_cube/inputs'
rec = np.load(cube_dir / 'reconstructed_exp_cube.npz', allow_pickle=True)
fit = np.load(cube_dir / 'full_cube_fit_reciprocal_MT.npz', allow_pickle=True)
ds = np.load(cube_dir / 'uniform_cube_16_strobe_20260505_230858.npz', allow_pickle=True)
bias = np.array([float(ds['k1_voltage']), float(ds['k2_voltage']), float(ds['k3_voltage'])])
V1, V2, V3 = rec['V1_raw'], rec['V2_raw'], rec['V3_raw']
O1, O2, O3 = rec['out1_raw'], rec['out2_raw'], rec['out3_raw']
c1, c2, c3 = fit['c1'], fit['c2'], fit['c3']
C1, C2, C3 = np.meshgrid(c1, c2, c3, indexing='ij')
MV1, MV2, MV3 = C3.ravel(), C2.ravel(), C1.ravel()
t_cube = fit['t_cube']
M1, M2, M3 = t_cube[...,0].ravel(), t_cube[...,2].ravel(), t_cube[...,1].ravel()

lo = np.array([min(O1.min(), M1.min()), min(O2.min(), M2.min()), min(O3.min(), M3.min())])
hi = np.array([max(O1.max(), M1.max()), max(O2.max(), M2.max()), max(O3.max(), M3.max())])
def shared_rgb(a, b, c):
    red = np.clip((a-lo[0])/(hi[0]-lo[0]), 0, 1)
    green = np.clip((b-lo[1])/(hi[1]-lo[1]), 0, 1)
    blue = np.clip((c-lo[2])/(hi[2]-lo[2]), 0, 1)
    return np.column_stack([red, green, blue])

def extinction_rgb(a, b, c, eps=1e-6):
    def n01(x): return (x-x.min())/(x.max()-x.min())
    r, g, b = n01(a), n01(b), n01(c)
    lr, lg, lb = [10*np.log10(x+eps) for x in (r,g,b)]
    qr = np.maximum(lr-np.maximum(lg,lb), 0)
    qg = np.maximum(lg-np.maximum(lr,lb), 0)
    qb = np.maximum(lb-np.maximum(lr,lg), 0)
    scale = max(qr.max(), qg.max(), qb.max())
    return np.clip(np.column_stack([qr, qg, qb])/scale, 0, 1), (qr, qg, qb)

exp_rgb = shared_rgb(O1, O2, O3)
model_rgb = shared_rgb(M1, M2, M3)
quality_rgb, quality_db = extinction_rgb(O1, O2, O3)
print(f'Experimental samples: {len(V1):,}; model grid points: {len(MV1):,}')

## Figures 4a–c — measured cube, fitted cube and measured extinction

In [ ]:
def brighten_rgb(rgb, gamma=0.55):
    # Display-only correction: underlying RGB data remain unchanged.
    return np.power(np.clip(rgb, 0, 1), gamma)

fig = plt.figure(figsize=(17, 5.5))
panels = [
    (V1, V2, V3, exp_rgb, 'Fig. 4a measured outputs'),
    (MV1, MV2, MV3, model_rgb, 'Fig. 4b fitted model'),
    (V1, V2, V3, quality_rgb, 'Fig. 4c measured extinction'),
]
for n, (x, y, z, color, title) in enumerate(panels, 1):
    ax = fig.add_subplot(1, 3, n, projection='3d')
    keep = np.linspace(0, len(x)-1, min(len(x), 5500), dtype=int)
    ax.scatter(x[keep], y[keep], z[keep], c=brighten_rgb(color[keep]), s=11, alpha=0.9, linewidths=0, depthshade=False)
    ax.view_init(elev=24, azim=42)
    ax.set(xlabel='V1 (V)', ylabel='V2 (V)', zlabel='V3 (V)', title=title)
fig.tight_layout()
print('Maximum measured extinction (dB):', ', '.join(f'{q.max():.2f}' for q in quality_db))

## Figure 4d — reciprocal-model slice through the three routing states

In [ ]:
R = np.asarray(ds['strobe_red_abs_v'], float)
G = np.asarray(ds['strobe_green_abs_v'], float)
B = np.asarray(ds['strobe_blue_abs_v'], float)
e1 = (G-R)/np.linalg.norm(G-R)
v2 = B-R-np.dot(B-R,e1)*e1
e2 = v2/np.linalg.norm(v2)
normal = np.cross(e1,e2)
model_points = np.column_stack([MV1, MV2, MV3])
distance = np.abs((model_points-R)@normal)
on_plane = distance < 0.5
u = (model_points[on_plane]-R)@e1
v = (model_points[on_plane]-R)@e2
model_quality_rgb, model_quality_db = extinction_rgb(M1, M2, M3)
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(u, v, c=brighten_rgb(model_quality_rgb[on_plane]), s=38, alpha=0.92, edgecolor='white', linewidth=0.25)
for point, label, color in [(R,'S1','#D62728'),(G,'S2','#2CA02C'),(B,'S3','#1F77B4')]:
    q = point-R
    ax.scatter(q@e1, q@e2, marker='*', s=260, color=color, edgecolor='black', linewidth=0.9, zorder=4)
    ax.annotate(label, (q@e1, q@e2), xytext=(8, 6), textcoords='offset points', weight='bold')
ax.set(aspect='equal', xlabel='e1 (V)', ylabel='e2 (V)', title='Fig. 4d model extinction slice')
fig.tight_layout()
print(f'{np.count_nonzero(on_plane):,} model points within 0.5 V of the routing-state plane')

## Figure 4e — time-domain sequential routing

In [ ]:
t_s = ds['scope_ch1_time']
dt = float(t_s[1]-t_s[0])
def fractional_shift(signal, delay_samples):
    source = (np.arange(len(signal))+delay_samples) % len(signal)
    i0 = np.floor(source).astype(int)
    fraction = source-i0
    return signal[i0]*(1-fraction)+signal[(i0+1)%len(signal)]*fraction
corrected = [fractional_shift(ds[f'scope_ch{i}_voltage'], ds['chirp_seed_d_out_s'][i-2]/dt) for i in (2,3,4)]
mask = (t_s >= 0) & (t_s <= 0.5e-6)
t_ns = t_s[mask]*1e9
outputs = [(x[mask]-x[mask].min())/(x[mask].max()-x[mask].min()) for x in corrected]
T = float(ds['strobe_t_cycle_ns']); width = T/5; alpha = 0.7
states = [np.asarray(ds['strobe_blue_awg_v'],float), np.asarray(ds['strobe_green_awg_v'],float)]
idle = -(np.asarray(ds['strobe_red_awg_v'],float)+np.asarray(ds['strobe_blue_awg_v'],float)+np.asarray(ds['strobe_green_awg_v'],float))/int(ds['strobe_n_idle_slots'])
states += [idle, idle, np.asarray(ds['strobe_red_awg_v'],float)]
phase = 10.1
cycle_t = (t_ns-phase)%T
slot = np.clip(np.floor(cycle_t/width).astype(int),0,4)
within = cycle_t-slot*width
fade = 0.5*(1-np.cos(np.pi*np.clip((within-(1-alpha)*width)/(alpha*width),0,1)))
K = np.empty((3,len(t_ns)))
for channel in range(3):
    current = np.array([s[channel] for s in states])
    following = np.roll(current,-1)
    K[channel] = current[slot]+(following[slot]-current[slot])*fade
fig, axes = plt.subplots(2,1,figsize=(12,6),sharex=True)
for values, style, label in zip(K, ['-','--',':'], ['V1','V2','V3']): axes[0].plot(t_ns, values, style, label=label)
for values, color, label in zip(outputs, ['tab:red','tab:green','tab:blue'], ['P1','P2','P3']): axes[1].plot(t_ns, values, color=color, label=label)
axes[0].set(ylabel='Voltage (V)', title='Fig. 4e reconstructed drive and measured outputs'); axes[0].legend(ncol=3)
axes[1].set(xlabel='Time (ns)', ylabel='Normalized power'); axes[1].legend(ncol=3)
fig.tight_layout()
print(f'{len(t_ns):,} displayed time samples; delay corrections (samples): {ds["chirp_seed_d_out_s"]/dt}')

## Interpretation

The plots above display the saved numerical values with simple styling.